# Breakdown of the Solution

## What the Current Code Does
The code I provided earlier:

- **Loads Models**:
  - **Phi-3**: Student model to solve geometry problems (text-only).
  - **Open LLaMA 7B & OPT-1.3B**: Teacher models to evaluate Phi-3’s answers.
  - **CLIP**: Image embedding model to process `image_diagram.png` and generate a basic text description (e.g., "A geometry diagram with a triangle").
- **Processes Data**:
  - Reads `data.json` (question, choices, correct answer) and `logic_form.json` (text constraints).
  - Uses CLIP to convert `image_diagram.png` into a simple JSON string (e.g., `{"diagram_description": "A geometry diagram with a triangle"}`).
- **Generates Answers**:
  - Feeds Phi-3 a prompt with the question, text constraints, and CLIP’s image description.
  - Phi-3 produces a detailed solution (e.g., "Final Answer: 58.7").
- **Maps to Choices**:
  - Extracts Phi-3’s raw answer (e.g., "58.7") and maps it to the closest option in `choices` (e.g., "60" from ["30", "60", "120", "240"]).
- **Evaluates & Corrects**:
  - Compares the mapped answer to `correct_solution` (e.g., "60").
  - Teachers evaluate Phi-3’s full response (correct/incorrect).
  - If Phi-3 is wrong or teachers disagree, it uses `correct_solution` for correction.
- **Trains Phi-3**:
  - Collects feedback for all 2101 problems and fine-tunes Phi-3 to improve its geometry skills.

**In Short**: It uses CLIP to give Phi-3 a basic hint about the diagram (e.g., "triangle"), solves the problem with text data, maps the answer to `choices`, and trains Phi-3 using the correct answers from `data.json`.

## What Adding `pytesseract` Does
`Pytesseract` is an OCR (Optical Character Recognition) tool that extracts text from images. If we replace CLIP with `pytesseract`, here’s what changes:

- **Image Processing**:
  - Instead of a vague description (e.g., "A geometry diagram with a triangle"), `pytesseract` reads actual text from `image_diagram.png` (e.g., "Triangle ABC, AB = 13, BC = 13, AC = 10").
  - Output becomes a JSON string like: `{"diagram_description": "Triangle ABC, AB = 13, BC = 13, AC = 10"}`.
- **Phi-3 Input**:
  - Phi-3 gets precise diagram details (e.g., side lengths) instead of a generic hint, improving its ability to solve accurately.
- **Accuracy**:
  - Answers should be closer to `correct_solution` because Phi-3 has more specific data to work with.
  - Mapping to `choices` (e.g., "60") becomes more reliable.

**In Short**: `Pytesseract` extracts detailed text from the image (e.g., labels, numbers), giving Phi-3 better context to generate answers, which are then mapped to `choices` and used to train it.

## Updated Code with `Pytesseract`

```python
# Install required libraries
!apt-get install -y tesseract-ocr
!pip install accelerate transformers datasets pandas pytesseract torch torchvision

import torch
import gc
import os
import json
import re
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from datasets import Dataset
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from PIL import Image
import pytesseract

# [Previous model loading code for Phi-3, Open LLaMA, OPT-1.3B remains the same]

def get_image_description(image_path):
    try:
        image = Image.open(image_path).convert("RGB")
        text = pytesseract.image_to_string(image)
        return json.dumps({"diagram_description": text.strip()})
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return json.dumps({"diagram_description": "Unknown diagram"})

# [Data loading code remains similar, just replace CLIP with pytesseract]
geometry_data = []
for idx, subfolder in enumerate(train_subfolders):
    data_file = os.path.join(subfolder, "data.json")
    logic_file = os.path.join(subfolder, "logic_form.json")
    image_file = os.path.join(subfolder, "image_diagram.png")
    
    if os.path.exists(data_file) and os.path.exists(logic_file) and os.path.exists(image_file):
        try:
            with open(data_file, 'r') as f:
                data_json = json.load(f)
            with open(logic_file, 'r') as f:
                logic_json = json.load(f)
            
            question = data_json.get("problem_text", "Unknown question")
            choices = data_json.get("choices", [])
            answer_idx = ord(data_json.get("answer", "A")) - ord("A")
            correct_solution = choices[answer_idx] if choices and 0 <= answer_idx < len(choices) else "Unknown solution"
            
            text_logic = logic_json.get("text_logic_form", ["Unknown task"])[0]
            diagram_logic = " ".join(logic_json.get("diagram_logic_form", []))
            image_json = get_image_description(image_file)
            
            geometry_data.append({
                "question": question,
                "text_logic": text_logic,
                "diagram_logic": diagram_logic,
                "choices": choices,
                "correct_solution": correct_solution,
                "image_json": image_json
            })
        except Exception as e:
            print(f"Error processing subfolder {subfolder}: {e}")

# [Rest of the code—prompt, Phi-3 generation, mapping, evaluation, training—remains the same]

# Summary
**CLIP Version**: Gives Phi-3 a basic image hint (e.g., "triangle"), trains it on text data with some diagram context.
**Pytesseract Version**: Extracts detailed text from the image (e.g., side lengths), making Phi-3’s answers more accurate and training more effective.
The pytesseract version should work better because it provides precise diagram details. Run this in your Kaggle notebook and share the output—I’ll tweak it further if needed! Which version do you want to try first?